In [1]:
# imports

# show rich formats 
from IPython.display import display, Markdown

# get python to interact with openai
from openai import OpenAI

# use personal openai key
import os
from dotenv import load_dotenv

# check load_dotenv works - should come back 'True'
# load_dotenv()

# make a destination 'resumes' directory for the work

os.makedirs("resumes", exist_ok=True)

# use python to format markdown to html
from markdown import markdown

#### because we're sending this out as a resume, we want it to be in a .pdf format. 
#### that means we'll have to use a library called weasy... 
##### <i><b>< record scratch ></i></b>
#### nope. sorry. 
#### not using weasyprint. lining weasyprint libraries up with each particular python environmet and setting / dedicating kernals still causes nightmares of 'public-speaking-in-underpants' proportions. surely a powerful tool, but it's got the playfulness and ease of use of a rabid porcupine. <i>no grazie</n>.
#### instead, going with pdfkit. working in html, so it'll do the job.
##### * note: pdfkit works using wkhtmltopdf, which <i>in very simple terms</i> converts a webpage to a pdf file. please see [reference.txt](https://github.com/npj210mlk/jobapp_prompter/blob/main/requirements.txt) in the repo for installation instructions.


In [2]:
# import pdfkit

# # test
# pdfkit.from_string("<h1>It should be called 'QueezyPrint,' amirite?</h1>", "output.pdf")

#### ha! apparently, jupyter agrees - came back 'True'

***
#### with the libraries imported we can now break the project down into four (4) steps:
##### 1.) open and read the markdown resume file
#####     * see requirements notes
##### 2.) input the desired job description
##### 3.) template some prompt engineering
##### 4.) convert markdown to html
##### 5.) convert html to pdf
***
*** 
#### Step 1: Open and Read the Markdown Resume
****

In [4]:
# open resume and read it
with open("/Users/nicholasjoseph/Desktop/nj_tam_resume.md", "r", encoding = "utf-8") as resume_file:
    resume_string = resume_file.read()

# check to see if it worked:
# display(Markdown(resume_string))

# markdown resume displays as expected.

***
#### Step 2: Input the Desired Job Description
***

In [5]:
# job description will be from anywhere, so input is used to pause the program until we find one to copy/paste into this variable 
job_desc_string = input()

 Data Engineer Job Title:Data Engineer (ONSITE San Antonio, TX) Location:San Antonio, TX Duration:05 May 2025 - 01 May 2026 Travel Type:On-site (no expenses)  Job Requirements:  1. SQL 2. Tableau 3. Python 4. Snowflake


***
#### Step 3: Template Some Prompt Engineering
##### - this involves dealing with ChatGPT, particularly the ChatGPT 4o-mini pre-trained transformer.
##### <u>a couple of things about the ChatGPT 4o-mini engine (model)</u>:
#####     a.) it is the smaller, more affordable version of the massive GPT-4o engine available to developers; and
#####     b.) because its focus is on text classificaton / prediction, it is purely a "decoder-only" type transformer
#### The idea is to have ChatGPT create the prompt to respond to the likely Applicant Tracking System being used by the job poster.
##### Reason being, machines talk to machines better than we can. 
***

In [6]:
# making up a random "lambda" function to create the variable "prompt_template"

# the text is what I would be putting into ChatGPT were I to do this one job at a time - prompt engineering

prompt_template = lambda resume_string, job_desc_string : f"""
### Role: 
You are a professional resume optimization expert, tailoring my resume to fit specific job descriptions. 
You know my job preferences include collaborating with people, and helping businesses get the most out of their data.
Your goal is to optimize my resume and provide actionable suggestions for improvement to align with the target role.

### Guidelines:
1. **Relevance**:
    - Prioritize the particular skills and experiences I have with what is **most relevant to the job position**.
    - De-emphasize or even completely remove irrelevant details to ensure a **concise** and **targeted** resume.
    - Limit work experience section to 2-3 most relevant roles
    - Limit bullet points under each role to 2-3 most relevant impacts
    - Select only the core competencies most relevant to the job description

2. **Action-Driven Results**:
    - Choose **strong action verbs** and **quantifiable results** (eg: percentages, revenues, efficiency improvement, etc.)

3. **Keyword Optimization**:
    - Integrate **keywords** and phrases from the job description naturally to optimize for Applicant Tracking Systems (ATS)

4. **Additional Suggestions*** *(if gaps exist)*:
    - If the resume does not fully align with the job description, suggest:
        a.) **Additional technical or soft skills** that I could add to make my profile stronger.
        b.) **Certifications or courses** I have (or could pursue) that would bridge the gap(s).
        c.) **Project ideas or experiences** that would better align with the role.

5.) **Formatting**:
    - Ouptut the tailored resume in **clean Markdown format**.
    - Include an **"Additional Suggestions"** section at the end with actionable improvement recommendations.

---

## Input:
- **My resume**:
{resume_string}

- **The Job Description**:
{job_desc_string}

---

### Output:
1. **Tailored Resume**:
    - A resume in **Markdown format** that emphasizes relevant experience, skills, and achievements.
    - Incorporates job description **keywords** to optimize for ATS.
    - Uses confident language and is no longer than **one page**.

2. **Additional Suggestions** *(if applicable)*:
    - List **skills** that could strengthen alignment with the role.
    - Recommend **certifications or courses** to pursue.
    - Suggest **specific projects or experiences** to develop.
"""

In [7]:
# set the prompt variable for our ChatGPT message roles
prompt = prompt_template(resume_string, job_desc_string)

***
#### Step 4: Generate the Resume with GPT-4o-mini
##### - Make the api call and tell GPT to write the resume using the prompt from above
***

In [11]:
# set up api client
open_apikey = os.environ.get("openapi_apikey")
    
client = OpenAI(api_key = open_apikey)

# make the call

# set response variable to hold the all info we get back
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    
    # set our roles up - think of casting a play: "Today, the role of the Expert Resume Writer will be played by the system."
    messages = [
        {"role" : "system", "content" : "Expert Resume Writer"},
        {"role" : "user", "content" : prompt}
    ],
    
    # give it some creative license 
    temperature = 0.7
)

# get our response
response_string = response.choices[0].message.content

#### Or, you can use Google's 'gemini-pro' engine to do the same thing. 
##### <u>Two reasons you may want to try this</u>:
##### 1.) It's a good chance to compare how well each of the engines responds; and 
##### 2.) You may exceed one rate limit (like I did), and need another service to continue moving forward.
##### <i>Either way</i>, just be sure to comment out the above code block for ChatGPT.

In [12]:
# import os
# import google.generativeai as genai

# # set up api key
# gemini_api_key = os.environ.get("GEMINI_API_KEY")
# genai.configure(api_key=gemini_api_key)

# # select the Gemini model you want to use
# model = genai.GenerativeModel('gemini-pro') # Or 'gemini-pro-vision' if your prompt involves images

# # make the call

# # set response variable to hold the all info we get back
# response = model.generate_content(
#     contents=[
#         {
#             "role": "user", # In Gemini, the system prompt is often integrated into the user prompt
#             "parts": [f"You are an Expert Resume Writer. Please optimize the following resume: {prompt}"]
#         }
#     ],
#     generation_config={
#         "temperature": 0.7
#     }
# )

# # get our response
# response_string = response.text

***
#### Step 5: Show Us the Results
***

In [13]:
# separate new resume from improvements AI suggests at 'Additional Suggestions'
response_list = response_string.split("## Additional Suggestions")

In [14]:
display(Markdown(response_list[0]))

# Nicholas Joseph  
**Data Engineer | Product & Data Strategist**  
San Antonio, TX | Remote | Hybrid  
nickpjoseph210@gmail.com | (210) 771-9853  
[LinkedIn](https://www.linkedin.com/in/nickjosephsanantonio/) | [GitHub](https://github.com/npj210mlk)  

---

## Career Summary  
Detail-oriented Data Engineer with over 7 years of experience in bridging business needs with technical solutions. Demonstrated expertise in SQL, Python, and cloud platforms including Snowflake and GCP. Proven track record of enhancing data accessibility and efficiency, driving strategic decision-making through data visualization, and fostering collaboration among cross-functional teams.

---

## Core Competencies  
- SQL & Database Management  
- Data Engineering & Cloud Solutions (Snowflake, GCP)  
- Data Visualization (Tableau)  
- Python Programming  
- Stakeholder Communication & Presentation  
- Agile Methodologies (Scrum/Kanban)  
- KPI & ROI Analysis  
- Technical Consulting & Solution Design  

---

## Professional Experience  

### Cloud Data Engineer / Technical Liaison  
**Spaulding Ridge** | Remote | 07/2022–06/2023  
- Led cloud migration initiatives for enterprise clients, achieving a 98.9% improvement in data access efficiency utilizing Snowflake and GCP.  
- Developed and delivered impactful presentations to executive stakeholders, translating engineering deliverables into actionable business insights.  
- Collaborated with sales and marketing teams to create training materials and collateral that enhanced product adoption and stakeholder engagement.

### Junior Data Engineer / Sprint Review Lead  
**Apexon** | Remote | 12/2020–03/2022  
- Spearheaded sprint presentations for a federal banking client, effectively bridging communication between development teams and stakeholders.  
- Supported cloud data migration efforts, ensuring compliance and alignment with regulatory standards while utilizing SQL for data management.  
- Co-founded a mentorship program that improved onboarding efficiency by over 30%, fostering a collaborative work environment.

### Business Development / Project Manager  
**NobleHands H & C** | San Antonio, TX | 10/2017–01/2020, 07/2023–Present  
- Identified customer friction points and delivered tailored data solutions, driving 200% Y.O.Y. revenue growth.  
- Acted as the primary liaison for customer accounts, ensuring alignment with client goals and satisfaction through effective communication.

---

## Certifications & Education  
- **Professional Scrum Master I** – scrum.org (2024)  
- **Project Management Professional (in progress)** – Technical Institute of America (2024)  
- **Data Science Certificate** – Codeup, San Antonio TX (2020)  
- **B.S. Biology** – Concordia University, Austin TX  

---

## Technical Skills  
**Languages & Tools:** Python, SQL, Tableau, Snowflake, NoSQL, dbt  
**Cloud Platforms:** GCP, AWS  
**Methodologies:** Agile (Scrum/Kanban)  
**Other:** Data Modeling, Client-Facing Dashboards, Data Analysis  

---



***
#### Step 6: Save the New Resume
##### Great. Hit several snags. You either need WeasyPrint installed in some capacity, or other tools I found (like 'Grip') are difficult to automate and test on MacOS. 
##### Looks like the programming gods have spoken: we're going with WeasyPrint.
##### <center><span style ="color:red"><b><u>To Do This:</u></b></span><center>
##### 1.) completely uninstall existing WeasyPrint's existence from your machine with pip and brew uninstalls;
##### 2.) run a 'brew cleanup' just to wipe out any remnants;
##### 3.) form home directory in Terminal (I'm using Mac), type 'brew install cairo pango gdk-pixbuf libffi' - these are the native Weasyprint dependencies;
##### 4.) set your environment variables in your profile (for me, my ~/.zshrc file) to the following:
###### export DYLD_LIBRARY_PATH=/opt/homebrew/lib:$DYLD_LIBRARY_PATH
###### export LIBRARY_PATH="/opt/homebrew/lib:$LIBRARY_PATH"
###### export PKG_CONFIG_PATH="/opt/homebrew/lib/pkgconfig:/opt/homebrew/share/pkgconfig"
###### export PATH="/opt/homebrew/bin:$PATH"
##### 5.) restart the terminal;
##### 6.) navigate to your project folder;
##### 7.) type 'pip install weasyprint markdown'; and (finally)
##### 8.) open your notebook from your project file with 'jupyter notebook'
***

In [15]:
# Markdown was already imported earlier
# import WeasyPrint and its HTML abilities
from weasyprint import HTML

# save it as PDF
output_pdf_file = "resumes/nick_joseph_tam_resume.pdf"

# convert the Markdown file to HTML
html_content = markdown(response_list[0])

# now take that HTML and convert it into a PDF and save
HTML(string=html_content).write_pdf(output_pdf_file)

In [17]:
# let's save the new Markdown file, too
markdown_output = "resumes/nickjoseph_dataengineer_resume_markdown.md"

with open(markdown_output, "w", encoding = "utf-8") as file:
    file.write(response_list[0])

***
#### Step 7: Display Suggestions for Improvement
##### Because we split "response_string" earlier, it was split at "Additional Suggestions," giving two items "response_list."
##### The first item ([0]) is the resume ChatGPT wrote with our prompt.
##### The second item ([1]) are the suggestions ChatGPT offers to make our resume stronger
***

In [18]:
# show us how to make the resume stronger
display(Markdown(response_list[1]))

  
- **Skills to Consider Adding**: Familiarity with data warehousing concepts, experience with ETL processes, and advanced SQL skills could enhance your alignment with the role.
- **Recommended Certifications**: Pursuing a certification specifically in Snowflake or Advanced SQL could strengthen your profile further.
- **Project Ideas**: Consider developing a personal or open-source project that involves building a data pipeline using Snowflake and Python, integrating various data sources, and visualizing the results in Tableau. This can showcase your technical skills while providing tangible results to discuss in interviews.

In [ ]:
# # markdown is already imported
# from weasyprint import HTML

In [ ]:
# from markdown2pdf import convert

# markdown_resume = display(Markdown(response_list[0]))

# convert(markdown_resume, "nj_resume_in_pdf.pdf")